CNN 

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

# ==========================================
# LOAD AUGMENTED CSV
# ==========================================

df = pd.read_csv(
    r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented.csv"
)

# ==========================================
# AUGMENTED DATASET DIRECTORY
# ==========================================

train_dir = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented"

X = []
y = []

# ==========================================
# LOAD IMAGES
# ==========================================

for _, row in df.iterrows():

    img_path = os.path.join(
        train_dir,
        str(row["Category"]),
        str(row["Id"]) + ".png"
    )

    img = Image.open(img_path).convert("L")

    # normalize
    img = np.array(img) / 255.0

    X.append(img)
    y.append(row["Category"])

# ==========================================
# CONVERT TO NUMPY
# ==========================================

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

# ==========================================
# RESHAPE FOR CNN
# ==========================================

X = X.reshape(-1, 32, 32, 1)

print("CNN shape:", X.shape)

# ==========================================
# TRAIN / VALIDATION SPLIT
# ==========================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================================
# CNN MODEL
# ==========================================

model = Sequential([

    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(32,32,1)
    ),

    MaxPooling2D((2,2)),

    Conv2D(
        64,
        (3,3),
        activation='relu'
    ),

    MaxPooling2D((2,2)),

    Flatten(),

    Dense(128, activation='relu'),

    Dropout(0.5),

    Dense(10, activation='softmax')
])

# ==========================================
# COMPILE
# ==========================================

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ==========================================
# TRAIN
# ==========================================

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val)
)

# ==========================================
# VALIDATION ACCURACY
# ==========================================

val_loss, val_acc = model.evaluate(X_val, y_val)

print("Validation Accuracy:", val_acc)

# ==========================================
# LOAD TEST SET
# ==========================================

X_test = []
names = []

test_dir = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\test\test"

for file in sorted(os.listdir(test_dir)):

    img_path = os.path.join(test_dir, file)

    img = Image.open(img_path).convert("L")

    img = np.array(img) / 255.0

    X_test.append(img)
    names.append(file)

X_test = np.array(X_test)

# reshape for CNN
X_test = X_test.reshape(-1, 32, 32, 1)

# ==========================================
# PREDICTIONS
# ==========================================

preds = model.predict(X_test)

pred_labels = np.argmax(preds, axis=1)

# ==========================================
# SUBMISSION CSV
# ==========================================

submission = pd.DataFrame({
    "Id": names,
    "Category": pred_labels
})

submission["Id"] = submission["Id"].str.replace(
    ".png",
    "",
    regex=False
)

submission.to_csv("submission_cnn.csv", index=False)

print("submission_cnn.csv saved")